In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [9]:
#header_silver = pd.read_csv(r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Silver-Layer\header_silver.csv')
#container_silver = pd.read_csv(r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Silver-Layer\container_silver.csv')
hazmat_class_silver = pd.read_csv(r'C:\Users\CY522SH\OneDrive - EY\Desktop\Project-1\Silver-Layer\hazmat_class_silver.csv')


In [10]:
#header_ids = header_silver.copy()
#container_ids = container_silver.copy()
hazmat_class_ids = hazmat_class_silver.copy()

In [11]:
def save_csv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)


In [20]:
carrier_dim = (
    header_silver[["carrier_code"]]
    .dropna()
    .drop_duplicates()
    .sort_values("carrier_code")
    .reset_index(drop=True)
)

carrier_dim["carrier_id"] = range(1, len(carrier_dim) + 1)
carrier_dim = carrier_dim[["carrier_id", "carrier_code"]]

save_csv(carrier_dim, "Lookups/carrier_dim.csv")

header_ids = (
    header_ids
    .merge(carrier_dim, on="carrier_code", how="left")
    .drop(columns=["carrier_code"])
)

In [21]:
manifest_unit_dim = (
    header_silver[["manifest_unit"]]
    .dropna()
    .drop_duplicates()
    .sort_values("manifest_unit")
    .reset_index(drop=True)
)

manifest_unit_dim["manifest_unit_id"] = range(1, len(manifest_unit_dim) + 1)
manifest_unit_dim = manifest_unit_dim[["manifest_unit_id", "manifest_unit"]]

save_csv(manifest_unit_dim, "Lookups/manifest_unit_dim.csv")

header_ids = (
    header_ids
    .merge(manifest_unit_dim, on="manifest_unit", how="left")
    .drop(columns=["manifest_unit"])
)

In [22]:
vessel_dim = (
    header_silver[["vessel_name", "vessel_country_code"]]
    .dropna(subset=["vessel_name"])
    .drop_duplicates()
    .sort_values("vessel_name")
    .reset_index(drop=True)
)

vessel_dim["vessel_id"] = range(1, len(vessel_dim) + 1)
vessel_dim = vessel_dim[["vessel_id", "vessel_name", "vessel_country_code"]]
save_csv(vessel_dim, "Lookups/vessel.csv")

header_ids = header_ids.merge(
    vessel_dim,
    on=["vessel_name", "vessel_country_code"],
    how="left"
).drop(columns=["vessel_name", "vessel_country_code"])

In [12]:
print(header_silver.columns.tolist())

['identifier', 'carrier_code', 'vessel_country_code', 'vessel_name', 'port_of_unlading', 'foreign_port_of_lading_qualifier', 'estimated_arrival_date', 'manifest_quantity', 'manifest_unit', 'weight', 'weight_unit', 'measurement', 'measurement_unit', 'record_status_indicator', 'place_of_receipt', 'port_of_destination', 'foreign_port_of_destination', 'foreign_port_of_destination_qualifier', 'conveyance_id_qualifier', 'conveyance_id', 'in_bond_entry_type', 'mode_of_transportation', 'second_party', 'actual_arrival_date']


In [23]:
port_cols = [
    "port_of_unlading",
    "port_of_destination",
    "foreign_port_of_destination"
]

port_dim = (
    header_silver[port_cols]
    .melt(value_name="port_name")["port_name"]
    .dropna()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
    .to_frame()
)

port_dim["port_id"] = range(1, len(port_dim) + 1)
port_dim = port_dim[["port_id", "port_name"]]

save_csv(port_dim, "Lookups/port.csv")

header_ids = header_silver.copy()

for col, fk in [
    ("port_of_unlading", "unlading_port_id"),
    ("port_of_destination", "destination_port_id"),
    ("foreign_port_of_destination", "foreign_destination_port_id")
]:
    header_ids = header_ids.merge(
        port_dim,
        left_on=col,
        right_on="port_name",
        how="left"
    ).rename(columns={"port_id": fk}).drop(columns=[col, "port_name"])

In [24]:
measurement_unit_dim = (
        header_silver[["measurement_unit"]]
        .dropna()
        .drop_duplicates()
        .sort_values("measurement_unit")
        .reset_index(drop=True)
)

measurement_unit_dim["measurement_unit_id"] = range(1, len(measurement_unit_dim) + 1)
measurement_unit_dim = measurement_unit_dim[["measurement_unit_id", "measurement_unit"]]

save_csv(measurement_unit_dim, "Lookups/measurement_unit.csv")

header_ids = (
        header_ids
        .merge(measurement_unit_dim, on="measurement_unit", how="left")
        .drop(columns=["measurement_unit"])
)

In [27]:
seal_info_dim = (
    container_silver[
        ["container_number", "seal_number_1", "seal_number_2"]
    ]
    .melt(
        id_vars=["container_number"],
        var_name="seal_col",
        value_name="seal_number"
    )
    .dropna(subset=["seal_number"])
)

seal_info_dim["seal_sequence"] = (
    seal_info_dim["seal_col"]
    .str.extract(r"(\d+)")  # Extract the number from 'seal_number_1' or 'seal_number_2'
    .astype(int)
)

seal_info_dim = (
    seal_info_dim
    .drop(columns=["seal_col"])
    .drop_duplicates()
    .sort_values(["container_number", "seal_sequence"])
    .reset_index(drop=True)
)

seal_info_dim["seal_id"] = range(1, len(seal_info_dim) + 1)

seal_info_dim = seal_info_dim[["seal_id", "container_number", "seal_sequence", "seal_number"]]

save_csv(seal_info_dim, "Lookups/seal_info.csv")

primary_seal = seal_info_dim.query(
    "seal_sequence == 1"
)[["container_number", "seal_id"]]

container_ids = (
    container_ids
    .merge(primary_seal, on="container_number", how="left")
    .rename(columns={"seal_id": "primary_seal_id"})
)

In [28]:
mode_of_transport_dim = (
    header_silver[["mode_of_transportation"]]
    .dropna()
    .drop_duplicates()
    .sort_values("mode_of_transportation")
    .reset_index(drop=True)
)

mode_of_transport_dim["mode_of_transportation_id"] = range(1, len(mode_of_transport_dim) + 1)

mode_of_transport_dim = mode_of_transport_dim[["mode_of_transportation_id", "mode_of_transportation"]]

save_csv(mode_of_transport_dim, "Lookups/mode_of_transportation.csv")

header_ids = (
    header_ids
    .merge(mode_of_transport_dim, on="mode_of_transportation", how="left")
    .drop(columns=["mode_of_transportation"])
)

In [30]:
load_status_dim = (
    container_silver[["load_status"]]
    .dropna()
    .drop_duplicates()
    .sort_values("load_status")
    .reset_index(drop=True)
)

load_status_dim["load_status_id"] = range(1, len(load_status_dim) + 1)
load_status_dim = load_status_dim[["load_status_id", "load_status"]]

save_csv(load_status_dim, "Lookups/load_status.csv")

load_status_map = \
    load_status_dim.set_index("load_status")["load_status_id"]

container_ids = \
    container_ids["load_status"].map(load_status_map)

container_ids.drop(columns=["load_status"], inplace=True)

In [12]:
hazmat_type_dim = (
    hazmat_class_silver[["hazmat_sequence_number", "hazmat_classification"]]
    .dropna()
    .drop_duplicates()
    .sort_values("hazmat_sequence_number")
    .reset_index(drop=True)
)



save_csv(hazmat_type_dim, "Lookups/hazmat_type.csv")

hazmat_class_ids = (
    hazmat_class_ids
    .merge(hazmat_type_dim, on="hazmat_classification", how="left")
).drop(columns=["hazmat_classification"])

In [14]:
hazmat_assignment_dim = (
    hazmat_class_silver[["container_number"]]
    .dropna(subset=["container_number"])
    .drop_duplicates()
    .sort_values(["container_number"])
    .reset_index(drop=True)
)

hazmat_assignment_dim["hazmat_assignment_id"] = range(1, len(hazmat_assignment_dim) + 1)


hazmat_assignment_dim = hazmat_assignment_dim[
    ["hazmat_assignment_id", "container_number"]
]

save_csv(hazmat_assignment_dim, "Lookups/hazmat_assignment.csv")

hazmat_class_ids = (
    hazmat_class_ids
    .merge(
        hazmat_assignment_dim,
        on=["container_number"],
        how="left"
    )
)

In [5]:
shipment_container_dim = (
    container_silver[["identifier", "container_number"]]
    .dropna()
    .drop_duplicates()
    .sort_values(["identifier", "container_number"])
    .reset_index(drop=True)
)

shipment_container_dim.head()

,identifier,container_number
0,201801010,FCIU9250931
1,201801011,EITU1595313
2,201801012,FCIU9250931
3,201801013,BMOU5389685
4,201801014,EMCU5289450


In [8]:
save_csv(shipment_container_dim, "Lookups/shipment_container.csv")

In [33]:

save_csv(header_ids, "Lookups/header_with_ids.csv")